In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os 
from scipy.optimize import curve_fit
from scipy.integrate import quad

storm_directory = 'data/constituents'
storms = {}
for filename in os.listdir(storm_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(storm_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        storms[key] = df                                    # store the data frame in the dictionary

shear_stress = pd.read_csv('data/shear_stress/average_total_shear_stress_corrected.csv', parse_dates=['datetime'], index_col='datetime')
sonde_downstream = pd.read_csv('data/sonde_data/sonde_down_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')
sonde_upstream = pd.read_csv('data/sonde_data/sonde_up_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')


Process and Merge Data

In [2]:
# average rows with duplicate timestamps on sonde data (second-resolution data)
sonde_downstream = sonde_downstream.groupby(level=0).mean(numeric_only=True)
sonde_upstream = sonde_upstream.groupby(level=0).mean(numeric_only=True)

# re-sample shear stress and sonde data to 1 min intervals
shear_stress = shear_stress.resample('1min').interpolate()
sonde_downstream_resampled = sonde_downstream.resample('1min').interpolate()
sonde_upstream_resampled = sonde_upstream.resample('1min').interpolate()

In [3]:
# join shear stress data and the matching sonde data with the storm data
merged_storms = {}

for storm_name, storm_df in storms.items():
    if "down" in storm_name.lower():
        sonde_df = sonde_downstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    elif "up" in storm_name.lower():
        sonde_df = sonde_upstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    else:
        continue

    merged_storm_df = storm_df.join(shear_stress, how="left")
    merged_storm_df = merged_storm_df.join(sonde_df, how="left")
    merged_storms[storm_name] = merged_storm_df

# also join the sonde data with the shear stress data for the entire record 
sonde_downstream = shear_stress.join(sonde_downstream[["Turbidity FNU", "fDOM RFU"]], how="left")
sonde_upstream = shear_stress.join(sonde_upstream[["Turbidity FNU", "fDOM RFU"]], how="left")

In [4]:
merged_storms['st1_up']

,SS (uL/L),SSC (mg/L),SRP (mg/L),TP (mg/L),PP (mg/L),POC (mg/L),DOC (mg/L),shear_stress,Turbidity FNU,fDOM RFU
Date_Time,,,,,,,,,,
2021-07-23 14:30:00,0.000,4.409565,0.02625,0.03125,0.00,2.864641,1.754,68.223515,4.9100,16.1960
2021-07-23 14:54:00,92.060,68.981000,0.04200,0.09400,0.15,11.105057,4.522,83.757128,10.0920,16.5600
2021-07-23 15:02:00,95.690,124.475000,0.04500,0.12000,0.28,29.712121,4.436,87.915423,13.9256,16.9048
2021-07-23 16:13:00,403.515,359.220000,0.09000,0.36200,1.66,41.279351,12.442,126.540663,126.0844,19.1732
2021-07-23 16:58:00,124.040,86.111000,0.08300,0.20800,0.38,9.524065,12.977,129.045967,43.2064,31.4764
2021-07-23 17:58:00,65.300,37.228000,0.06700,0.13200,0.13,6.538644,13.108,110.251417,20.8056,37.2440
2021-07-23 20:13:00,42.870,30.435000,0.04800,0.08600,0.06,5.707480,12.073,90.089846,10.3608,38.9992


Hysteresis index calculation functions

In [5]:
## regression equations
# linear
def linear_func(Q, a, b):
    return a * Q + b
# logarithmic
def log_func(Q, a, b):
    return a * np.log(Q) + b
# exponential
def exp_func(Q, a, b):
    return a * np.exp(b * Q)

# split hydrograph into rising and falling limbs based on peak flow
def split_hydrograph(df, q_col):
    # if df empty or q_col has no valid values, return empty limbs
    if df.empty or df[q_col].dropna().empty:
        return df.copy(), df.copy()
    peak_time = df[q_col].idxmax()
    rising = df.loc[:peak_time].copy()
    falling = df.loc[peak_time:].copy()
    return rising, falling

# fit curves and calculate R²
def fit_best_curve(x, y):
    candidate_functions = {
        'linear': (linear_func, [1, 1]),
        'log': (log_func, [1, 1]),
        'exponential': (exp_func, [1, -0.01])
    }
    mask = np.isfinite(x) & np.isfinite(y)
    x = np.asarray(x[mask])
    y = np.asarray(y[mask])

    # minimum points
    if len(x) < 2:
        return None
    best_r2 = -np.inf
    best_result = None
    # try each function and keep the one with the best r2
    for func_name, (func, p0) in candidate_functions.items():
        try:
            # avoid invalid log fits
            if func_name == 'log' and np.any(x <= 0):
                continue
            popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
            y_pred = func(x, *popt) # predicted values
            # residuals
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            # avoid divide-by-zero
            if np.isclose(ss_tot, 0):
                r2 = np.nan
            else:
                r2 = 1 - (ss_res / ss_tot)
            # keep best fit
            if np.isfinite(r2) and r2 > best_r2:
                best_r2 = r2
                best_result = {
                    'function_name': func_name,
                    'function': func,
                    'params': popt,
                    'r2': r2
                }
        except Exception:
            continue
    return best_result

# Langlois 2025 H calculation
def compute_langlois_H(event_df, tau_col, constituent_col, r2_threshold=0.50, storm_name=None):
    # data cleanup
    df = event_df[[tau_col, constituent_col]].dropna()
    if df.empty or df[tau_col].dropna().empty or df[constituent_col].dropna().empty:
        print(f"No valid {tau_col} or {constituent_col} for {storm_name or 'unknown'}; skipping")
        return None

    rising, falling = split_hydrograph(df, tau_col)
    if rising.empty or falling.empty:
        print(f"No rising or falling limb for {storm_name or 'unknown'}; skipping")
        return None

    # shear stress overlap range
    tau_min = max(rising[tau_col].min(), falling[tau_col].min())
    tau_max = min(rising[tau_col].max(), falling[tau_col].max())

    # fit rising limb
    rise_fit = fit_best_curve(rising[tau_col].values, rising[constituent_col].values)
    if rise_fit is None:
        print(f"Could not fit rising limb in " f"{storm_name or 'unknown'} for {constituent_col}")
        return None
    # fit falling limb
    fall_fit = fit_best_curve(falling[tau_col].values, falling[constituent_col].values)
    if fall_fit is None:
        print(f"Could not fit falling limb in " f"{storm_name or 'unknown'} for {constituent_col}")
        return None

    # check fit quality with r2 threshold
    if (rise_fit['r2'] < r2_threshold) or (fall_fit['r2'] < r2_threshold):
            label = storm_name if storm_name is not None else "unknown storm"
            print(f"Poor fit for rising (R²={rise_fit['r2']:.2f}) or falling (R²={fall_fit['r2']:.2f}) limb in {label} for {constituent_col}")
    # integrated areas
    rise_area, _ = quad(lambda q: rise_fit["function"](q, *rise_fit["params"]), tau_min, tau_max)
    fall_area, _ = quad(lambda q: fall_fit["function"](q, *fall_fit["params"]), tau_min, tau_max)
    # hysteresis index
    H = rise_area / fall_area

    return {
        # hysteresis
        'H': H,
        # rising limb
        'rise_r2': rise_fit['r2'],
        'rise_function': rise_fit['function'],
        'rise_params': rise_fit['params'],
        'rise_area': rise_area,
        # falling limb
        'fall_r2': fall_fit['r2'],
        'fall_function': fall_fit['function'],
        'fall_params': fall_fit['params'],
        'fall_area': fall_area,
        # overlap range
        'tau_min': tau_min,
        'tau_max': tau_max,
        # point counts
        'n_rising': len(rising),
        'n_falling': len(falling)
    }

Calculate H for all events

In [6]:
all_results = []

for storm_name, storm_df in merged_storms.items():
    for constituent in ["SSC (mg/L)", "SRP (mg/L)", "TP (mg/L)", "PP (mg/L)", "POC (mg/L)", "DOC (mg/L)"]:
        if constituent not in storm_df.columns:
            continue

        result = compute_langlois_H(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

        if result is not None:
            result["storm"] = storm_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('HI_calculations/langlois_hysteresis_results.csv', index=False)

Poor fit for rising (R²=0.36) or falling (R²=0.85) limb in st1_down for SSC (mg/L)
Poor fit for rising (R²=0.35) or falling (R²=0.96) limb in st1_down for POC (mg/L)
Poor fit for rising (R²=0.46) or falling (R²=0.94) limb in st1_up for SSC (mg/L)
Poor fit for rising (R²=0.26) or falling (R²=0.94) limb in st1_up for POC (mg/L)
Poor fit for rising (R²=0.40) or falling (R²=0.99) limb in st2_down for SSC (mg/L)
Poor fit for rising (R²=0.40) or falling (R²=0.99) limb in st2_down for PP (mg/L)
Poor fit for rising (R²=0.32) or falling (R²=0.89) limb in st2_down for POC (mg/L)
Poor fit for rising (R²=0.75) or falling (R²=0.26) limb in st3_down for PP (mg/L)
Poor fit for rising (R²=0.16) or falling (R²=0.03) limb in st4_down for SSC (mg/L)
Poor fit for rising (R²=0.06) or falling (R²=0.71) limb in st4_down for POC (mg/L)
Poor fit for rising (R²=0.36) or falling (R²=0.52) limb in st4_up for SSC (mg/L)
Poor fit for rising (R²=0.14) or falling (R²=0.72) limb in st4_up for DOC (mg/L)
Could not fit 

C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y

Poor fit for rising (R²=0.49) or falling (R²=0.59) limb in st7_up for SSC (mg/L)
Poor fit for rising (R²=0.34) or falling (R²=0.01) limb in st7_up for POC (mg/L)
Poor fit for rising (R²=0.00) or falling (R²=0.80) limb in st7_up for DOC (mg/L)


### Plots

In [7]:
def plot_langlois_hysteresis(event_df, tau_col, constituent_col, r2_threshold=0.5, storm_name=None, 
                            out_dir='HI_calculations/langlois_plots', save=True, show=False):

    df = event_df[[tau_col, constituent_col]].dropna()
    if df.empty:
        return None
    rising, falling = split_hydrograph(df, tau_col)
    if rising.empty or falling.empty:
        return None
    
    # reuse the same fitting logic as the H calculation
    result = compute_langlois_H(
        event_df,
        tau_col=tau_col,
        constituent_col=constituent_col,
        r2_threshold=r2_threshold,
        storm_name=storm_name,
    )
    if result is None:
        return None

    tau_min = result["tau_min"]
    tau_max = result["tau_max"]
    if not np.isfinite(tau_min) or not np.isfinite(tau_max) or tau_min >= tau_max:
        return None

    tau_fit = np.linspace(tau_min, tau_max, 200)

    rise_params = result["rise_params"]
    fall_params = result["fall_params"]
    rise_r2 = result["rise_r2"]
    fall_r2 = result["fall_r2"]
    H = result["H"]

    rise_fit = result["rise_function"](tau_fit, *result["rise_params"])
    fall_fit = result["fall_function"](tau_fit, *result["fall_params"])

    # PLOT 
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # time series
    ax = axes[0]
    ax.plot(df.index, df[tau_col], color="tab:blue", linewidth=1.5, label=tau_col)
    ax.set_ylabel(tau_col, color="tab:blue")
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax.xaxis.set_major_locator(plt.MaxNLocator(8))
    ax2 = ax.twinx()
    ax2.plot(df.index, df[constituent_col], color="tab:red", linewidth=1.5, label=constituent_col)
    ax2.set_ylabel(constituent_col, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    ax.set_title("Event time series")

    # hysteresis loop
    ax = axes[1]
    ax.scatter(rising[tau_col], rising[constituent_col], label='Rising limb', color='tab:orange')
    ax.scatter(falling[tau_col], falling[constituent_col], label='Falling limb', color='tab:green')
    ax.plot(tau_fit, rise_fit, linewidth=2, label=f'Rising fit (R²={rise_r2:.2f})', color='tab:orange')
    ax.plot(tau_fit, fall_fit, linewidth=2, label=f'Falling fit (R²={fall_r2:.2f})', color='tab:green')

    ax.set_xlabel(tau_col)
    ax.set_ylabel(constituent_col)
    ax.set_title(f'H = {H:.2f}')
    ax.legend()

    # add a main title for the whole figure
    main_title = f"{storm_name} - {constituent_col} Hysteresis" if storm_name else f"{constituent_col} Hysteresis"
    plt.suptitle(main_title, fontsize=15)
    plt.tight_layout()

    if save:
        os.makedirs(out_dir, exist_ok=True)
        safe_name = f"{storm_name}_{constituent_col}_langlois.png".replace(" ", "_").replace("/", "_")
        fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    plt.close(fig)
    return result

In [8]:
all_results = []

for storm_name, storm_df in merged_storms.items():
    for constituent in ["SSC (mg/L)", "SRP (mg/L)", "TP (mg/L)", "PP (mg/L)", "POC (mg/L)", "DOC (mg/L)"]:
        if constituent not in storm_df.columns:
            continue

        plot_langlois_hysteresis(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

Poor fit for rising (R²=0.36) or falling (R²=0.85) limb in st1_down for SSC (mg/L)
Poor fit for rising (R²=0.35) or falling (R²=0.96) limb in st1_down for POC (mg/L)
Poor fit for rising (R²=0.46) or falling (R²=0.94) limb in st1_up for SSC (mg/L)
Poor fit for rising (R²=0.26) or falling (R²=0.94) limb in st1_up for POC (mg/L)
Poor fit for rising (R²=0.40) or falling (R²=0.99) limb in st2_down for SSC (mg/L)
Poor fit for rising (R²=0.40) or falling (R²=0.99) limb in st2_down for PP (mg/L)
Poor fit for rising (R²=0.32) or falling (R²=0.89) limb in st2_down for POC (mg/L)
Poor fit for rising (R²=0.75) or falling (R²=0.26) limb in st3_down for PP (mg/L)
Poor fit for rising (R²=0.16) or falling (R²=0.03) limb in st4_down for SSC (mg/L)
Poor fit for rising (R²=0.06) or falling (R²=0.71) limb in st4_down for POC (mg/L)
Poor fit for rising (R²=0.36) or falling (R²=0.52) limb in st4_up for SSC (mg/L)
Poor fit for rising (R²=0.14) or falling (R²=0.72) limb in st4_up for DOC (mg/L)
Could not fit 

C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


Poor fit for rising (R²=0.10) or falling (R²=0.45) limb in st5_up for DOC (mg/L)


C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


Poor fit for rising (R²=1.00) or falling (R²=0.00) limb in st6_up for PP (mg/L)


C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
C:\Users\huck4481\AppData\Local\Temp\ipykernel_17924\2399604567.py:44: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y

Poor fit for rising (R²=0.49) or falling (R²=0.59) limb in st7_up for SSC (mg/L)
Poor fit for rising (R²=0.34) or falling (R²=0.01) limb in st7_up for POC (mg/L)
Poor fit for rising (R²=0.00) or falling (R²=0.80) limb in st7_up for DOC (mg/L)


## Turbidity and fDOM hysteresis 

Separating storm by date and time

In [9]:
# define storm windows once
storm_windows = {
    "st1_down": ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st1_up":   ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st2_down": ("2022-08-03 15:00", "2022-08-03 19:00"),
    "st2_up":   ("2022-08-03 15:00", "2022-08-03 19:00"),
    "st3_down": ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st3_up":   ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st4_down": ("2023-07-29 13:00", "2023-07-30 10:30"),
    "st4_up":   ("2023-07-29 13:00", "2023-07-30 10:30"),
    "st5_down": ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st5_up":   ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st6_down": ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st6_up":   ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st7_down": ("2023-09-14 13:00", "2023-09-15 13:00"),
    "st7_up":   ("2023-09-14 13:00", "2023-09-15 13:00")
}

# build event dictionary directly (no function)
sonde_events = {}
down = sonde_downstream.sort_index()   
up = sonde_upstream.sort_index()      

for storm_name, (start, end) in storm_windows.items():
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    if "down" in storm_name.lower():
        src = down
    elif "up" in storm_name.lower():
        src = up
    else:
        continue
    sonde_events[storm_name] = src.loc[(src.index >= start) & (src.index <= end)].copy()

Calculate HI for sonde data

In [10]:
sonde_results = []

for storm_name, storm_df in sonde_events.items():
    for constituent in ["Turbidity FNU", "fDOM RFU"]:
        if constituent not in storm_df.columns:
            continue

        result = compute_langlois_H(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

        if result is not None:
            result["storm"] = storm_name
            result["constituent"] = constituent
            sonde_results.append(result)

sonde_results = pd.DataFrame(sonde_results)
sonde_results.to_csv('HI_calculations/langlois_sonde_hysteresis_results.csv', index=False)

Poor fit for rising (R²=0.05) or falling (R²=0.74) limb in st2_down for Turbidity FNU
Poor fit for rising (R²=0.53) or falling (R²=0.04) limb in st2_down for fDOM RFU
No valid shear_stress or Turbidity FNU for st2_up; skipping
No valid shear_stress or fDOM RFU for st2_up; skipping
No valid shear_stress or Turbidity FNU for st3_up; skipping
No valid shear_stress or fDOM RFU for st3_up; skipping
Poor fit for rising (R²=0.08) or falling (R²=0.18) limb in st4_down for fDOM RFU
Poor fit for rising (R²=0.43) or falling (R²=0.14) limb in st4_up for fDOM RFU
Poor fit for rising (R²=0.45) or falling (R²=0.91) limb in st5_down for Turbidity FNU
No valid shear_stress or Turbidity FNU for st5_up; skipping
No valid shear_stress or fDOM RFU for st5_up; skipping
Poor fit for rising (R²=0.71) or falling (R²=0.37) limb in st6_down for Turbidity FNU
Poor fit for rising (R²=0.88) or falling (R²=0.18) limb in st6_down for fDOM RFU
No valid shear_stress or Turbidity FNU for st6_up; skipping
No valid shear_

In [11]:
sonde_results

,H,rise_r2,rise_function,rise_params,rise_area,fall_r2,fall_function,fall_params,fall_area,tau_min,tau_max,n_rising,n_falling,storm,constituent
0,1.787826,0.737888,<function exp_func at 0x0000022C945CC9A0>,"[0.9770266222764296, 0.03900541279151098]",4168.472038,0.660763,<function log_func at 0x0000022C945CC860>,"[87.03353780183961, -363.9248392472115]",2331.587075,65.947067,133.064651,34,626,st1_down,Turbidity FNU
1,0.469250,0.624956,<function log_func at 0x0000022C945CC860>,"[16.14218727585002, -48.36605961605004]",1716.495026,0.669175,<function log_func at 0x0000022C945CC860>,"[85.82013537748632, -338.60495554792203]",3657.951513,65.947067,133.064651,34,626,st1_down,fDOM RFU
2,2.877064,0.779730,<function linear_func at 0x0000022C945CC900>,"[1.7967611873123768, -124.44498200195996]",3703.333349,0.971368,<function exp_func at 0x0000022C945CC9A0>,"[0.1775524870804954, 0.04357356879372031]",1287.191672,68.376213,133.471429,29,401,st1_up,Turbidity FNU
3,0.527705,0.588038,<function log_func at 0x0000022C945CC860>,"[12.128230694393611, -37.09368972339725]",1214.238552,0.650307,<function log_func at 0x0000022C945CC860>,"[47.8573189381949, -184.62646993905483]",2300.979540,68.376213,133.471429,29,401,st1_up,fDOM RFU
4,1.205008,0.050696,<function exp_func at 0x0000022C945CC9A0>,"[67.17002167105036, -0.02305129699244837]",100.194552,0.737063,<function log_func at 0x0000022C945CC860>,"[84.87796718441376, -345.8260042320126]",83.148479,64.042231,71.117322,16,26,st2_down,Turbidity FNU
5,0.761513,0.528259,<function log_func at 0x0000022C945CC860>,"[116.24776790446333, -461.06181251236944]",202.862622,0.038312,<function log_func at 0x0000022C945CC860>,"[8.251550065688019, 2.889848650574221]",266.394179,64.042231,71.117322,16,26,st2_down,fDOM RFU
6,1.119110,0.977780,<function exp_func at 0x0000022C945CC9A0>,"[0.0003511375967857438, 0.14686219779520338]",108.101941,0.931414,<function exp_func at 0x0000022C945CC9A0>,"[0.00019771649938258844, 0.1533759741786606]",96.596375,64.588449,74.728309,14,43,st3_down,Turbidity FNU
7,0.632417,0.933039,<function exp_func at 0x0000022C945CC9A0>,"[0.8727855094093959, 0.047037872392252406]",236.619042,0.597781,<function linear_func at 0x0000022C945CC900>,"[-0.4335529292354777, 67.09958320920782]",374.150574,64.588449,74.728309,14,43,st3_down,fDOM RFU
8,1.407495,0.875830,<function log_func at 0x0000022C945CC860>,"[167.29953166215446, -691.2219677388446]",357.485235,0.848765,<function log_func at 0x0000022C945CC860>,"[135.39462212318853, -561.9910029393799]",253.986784,66.121986,79.767296,20,100,st4_down,Turbidity FNU
9,0.685878,0.076854,<function exp_func at 0x0000022C945CC9A0>,"[5.33964086501436, 0.0052347616433221595]",106.763346,0.182629,<function log_func at 0x0000022C945CC860>,"[24.86298817154368, -95.2108990882725]",155.659346,66.121986,79.767296,20,100,st4_down,fDOM RFU


Plot

In [12]:
all_results = []

for storm_name, storm_df in sonde_events.items():
    for constituent in ["Turbidity FNU", "fDOM RFU"]:
        if constituent not in storm_df.columns:
            continue

        plot_langlois_hysteresis(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name,
            out_dir='HI_calculations/langlois_plots/sonde')

Poor fit for rising (R²=0.05) or falling (R²=0.74) limb in st2_down for Turbidity FNU
Poor fit for rising (R²=0.53) or falling (R²=0.04) limb in st2_down for fDOM RFU
Poor fit for rising (R²=0.08) or falling (R²=0.18) limb in st4_down for fDOM RFU
Poor fit for rising (R²=0.43) or falling (R²=0.14) limb in st4_up for fDOM RFU
Poor fit for rising (R²=0.45) or falling (R²=0.91) limb in st5_down for Turbidity FNU
Poor fit for rising (R²=0.71) or falling (R²=0.37) limb in st6_down for Turbidity FNU
Poor fit for rising (R²=0.88) or falling (R²=0.18) limb in st6_down for fDOM RFU
Poor fit for rising (R²=0.76) or falling (R²=0.22) limb in st7_up for fDOM RFU


In [13]:
sonde_events['st1_down']

,shear_stress,Turbidity FNU,fDOM RFU
datetime,,,
2021-07-23 12:30:00,63.685701,1.810,14.548
2021-07-23 12:31:00,63.699990,NaN,NaN
2021-07-23 12:32:00,63.714279,NaN,NaN
2021-07-23 12:33:00,63.728568,NaN,NaN
2021-07-23 12:34:00,63.742857,NaN,NaN
...,...,...,...
2021-07-28 11:11:00,65.966569,NaN,NaN
2021-07-28 11:12:00,65.961694,NaN,NaN
2021-07-28 11:13:00,65.956818,NaN,NaN
